# Anime Item-Based Collaborative Filtering

Day 2 baseline: explicit MyAnimeList ratings, sparse cosine item similarity, top-K similar titles, and user-aware Recall@K / NDCG@K evaluation. Raw ratings are limited to a configurable popular-item subset to keep similarity computation practical.

In [6]:
from pathlib import Path
import importlib
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'anime_recommender.py').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import anime_recommender
importlib.reload(anime_recommender)
from anime_recommender import (
    build_similarity_model,
    evaluate_holdout,
    evaluate_popularity_holdout,
    load_dataset,
    recommend_for_anime,
    split_user_holdout,
)

DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
ANIME_PATH = DATA_DIR / 'anime.csv'
RATINGS_PATH = DATA_DIR / 'rating.csv'
MAX_ITEMS = 750
MIN_ITEM_RATINGS = 20
K = 10

## Load and clean

Place Kaggle's `anime.csv` and `rating.csv` in the project `data/` folder. Expected rating fields are `user_id`, `anime_id`, and `rating`; anime metadata needs `anime_id` and `name` or `title`. The loader drops invalid ratings, averages duplicate user-item rows, joins metadata, and selects the most-rated items.

In [7]:
DATA_AVAILABLE = ANIME_PATH.is_file() and RATINGS_PATH.is_file()
if DATA_AVAILABLE:
    ratings = load_dataset(
        ANIME_PATH,
        RATINGS_PATH,
        max_items=MAX_ITEMS,
        min_item_ratings=MIN_ITEM_RATINGS,
    )
    catalog = ratings[['anime_id', 'title', 'genre', 'synopsis']].drop_duplicates('anime_id')
    print(f'{len(ratings):,} cleaned ratings across {len(catalog):,} anime')
    display(ratings.head())
else:
    print(f'Dataset not found. Add anime.csv and rating.csv to {DATA_DIR} and rerun this cell.')

4,097,576 cleaned ratings across 750 anime


,user_id,anime_id,rating,title,genre,synopsis
0,1,8074,10.0,Highschool of the Dead,"Action, Ecchi, Horror, Supernatural",
1,1,11617,10.0,High School DxD,"Comedy, Demons, Ecchi, Harem, Romance, School",
2,1,11757,10.0,Sword Art Online,"Action, Adventure, Fantasy, Game, Romance",
3,1,15451,10.0,High School DxD New,"Action, Comedy, Demons, Ecchi, Harem, Romance,...",
4,2,11771,10.0,Kuroko no Basket,"Comedy, School, Shounen, Sports",


## Build and evaluate

One rating of at least 7 is held out per eligible user. This creates a single known-relevant anime per evaluation case. Similarity is built from the remaining ratings only.

In [ ]:
from scipy.sparse import save_npz

if DATA_AVAILABLE:
    train, test = split_user_holdout(
        ratings, random_state=42, min_user_ratings=2, relevance_threshold=7
    )
    model = build_similarity_model(train, catalog)
    metrics, evaluation_details = evaluate_holdout(model, train, test, k=K)
    popularity_metrics, popularity_details = evaluate_popularity_holdout(train, test, k=K)
    metrics_table = pd.DataFrame([
        {'model': 'item_cosine', **metrics},
        {'model': 'popularity', **popularity_metrics},
    ])
    RESULTS_DIR.mkdir(exist_ok=True)
    ratings.to_csv(RESULTS_DIR / 'cleaned_ratings.csv.gz', index=False, compression='gzip')
    catalog.to_csv(RESULTS_DIR / 'anime_catalog.csv', index=False)
    save_npz(RESULTS_DIR / 'user_item_matrix.npz', model.user_item_matrix)
    save_npz(RESULTS_DIR / 'anime_similarity_matrix.npz', model.similarities)
    metrics_table.to_csv(RESULTS_DIR / 'baseline_metrics.csv', index=False)
    evaluation_details.to_csv(RESULTS_DIR / 'baseline_user_rankings.csv', index=False)
    popularity_details.to_csv(RESULTS_DIR / 'baseline_popularity_user_rankings.csv', index=False)
    print(f'Artifacts saved to {RESULTS_DIR}')
    display(metrics_table)
    display(evaluation_details.head())
else:
    print('Add the dataset files and run the load cell before evaluating.')

## Similar anime lookup

Change `SELECTED_ANIME_TITLE` to an exact title from the catalog. Matching is case-insensitive.

In [4]:
if DATA_AVAILABLE:
    SELECTED_ANIME_TITLE = catalog.iloc[0]['title']
    recommendations = recommend_for_anime(model, SELECTED_ANIME_TITLE, k=K)
    print(f'Recommendations similar to {SELECTED_ANIME_TITLE}:')
    display(pd.DataFrame(recommendations))
else:
    print('Add the dataset files and run the load cell before requesting recommendations.')

Recommendations similar to Highschool of the Dead:


,anime_id,title,similarity
0,11757,Sword Art Online,0.564319
1,6547,Angel Beats!,0.561285
2,11617,High School DxD,0.559006
3,9515,Highschool of the Dead: Drifters of the Dead,0.552774
4,10620,Mirai Nikki (TV),0.535389
5,6880,Deadman Wonderland,0.520627
6,226,Elfen Lied,0.518095
7,16498,Shingeki no Kyojin,0.516286
8,1535,Death Note,0.512002
9,8841,Kore wa Zombie Desu ka?,0.505022


## Baseline notes

On the 65,444 positive holdouts, item cosine reached Recall@10 0.117337 / NDCG@10 0.066312; popularity reached 0.107527 / 0.054898. The improvement is measurable but modest.

This holdout is a noisy proxy for item similarity: each user's seed is simply the first row in the retained training history, and the target is one other title rated at least 7. Those two titles may be unrelated. The randomized split is not temporal, raw-rating cosine does not correct for individual rating scales, and the 750-item cap excludes cold-start anime. Interpret these as baseline measurements for the filtered popular-item set, not production ranking quality.